# Camada Gold — ecommerce_pedidos

## Objetivo

Gerar métricas agregadas de Data Quality para consumo analítico e visualização em BI.

### Saídas esperadas

- `gold_dq_resumo_por_regra`
- `gold_dq_resumo_por_tabela`

### Fluxo

dq_monitoring_logs + Silver → agregações de qualidade → Gold Delta → Azure SQL

In [0]:
# =========================================================
# IMPORTS
# =========================================================

from pyspark.sql import functions as F

In [0]:
%run ./00_autenticação

In [0]:
# =========================================================
# CONFIGURAÇÃO DOS CAMINHOS
# =========================================================

storage_account = "internshipdatalake"
lake_container = "squad1"

silver_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"silver/ecommerce_pedidos"
)

dq_logs_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"dq_monitoring_logs"
)

gold_regra_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"gold/gold_dq_resumo_por_regra"
)

gold_tabela_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"gold/gold_dq_resumo_por_tabela"
)

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
# =========================================================
# LEITURA DAS FONTES
# =========================================================

df_logs = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(dq_logs_path)
)

df_silver = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(silver_path)
)

print("Logs de DQ:", df_logs.count())
print("Registros na Silver:", df_silver.count())

In [0]:
# =========================================================
# GOLD — RESUMO DE DQ POR REGRA
# =========================================================

df_gold_regra = (
    df_logs
    .filter(F.col("tabela") == "ecommerce_pedidos")
    .withColumn(
        "data",
        F.to_date(
            F.regexp_extract(
                "arquivo_origem",
                r"real-time-data/(\d{4}/\d{2}/\d{2})/",
                1
            ),
            "yyyy/MM/dd"
        )
    )
    .groupBy(
        "data",
        "tabela",
        "regra",
        "severidade"
    )
    .agg(
        F.sum("qtd_registros_falhos")
            .alias("qtd_registros_falhos"),

        F.sum("qtd_registros_total")
            .alias("qtd_registros_total")
    )
    .withColumn(
        "pct_falha",
        F.round(
            (
                F.col("qtd_registros_falhos")
                / F.col("qtd_registros_total")
            ) * 100,
            2
        )
    )
    .orderBy(
        "data",
        "regra"
    )
)

df_gold_regra.show(50, truncate=False)

In [0]:
# =========================================================
# VALIDAÇÃO — RESUMO POR REGRA
# =========================================================

print("Linhas na Gold por regra:", df_gold_regra.count())

df_gold_regra.select(
    F.sum(
        F.col("pct_falha").isNull().cast("int")
    ).alias("pct_falha_nulo")
).show()

In [0]:
# =========================================================
# GOLD — RESUMO DE DQ POR TABELA
# =========================================================

dq_flags = [
    "dq_id_pedido_invalido",
    "dq_id_cliente_nulo",
    "dq_status_invalido",
    "dq_valor_total_invalido",
    "dq_metodo_pagamento_invalido",
    "dq_data_status_invalida",
    "dq_frete_invalido",
    "dq_cancelado_com_rastreamento_ativo",
    "dq_entrega_prazo_invalido",
    "dq_endereco_entrega_invalido"
]

df_gold_tabela_base = (
    df_silver
    .withColumn(
        "registro_com_falha",
        F.greatest(
            *[F.col(c).cast("int") for c in dq_flags]
        )
    )
    .withColumn(
        "data",
        F.to_date(
            F.regexp_extract(
                "bronze_source_file",
                r"real-time-data/(\d{4}/\d{2}/\d{2})/",
                1
            ),
            "yyyy/MM/dd"
        )
    )
    .withColumn(
        "hora",
        F.substring(
            F.regexp_extract(
                "bronze_source_file",
                r"real-time-data/\d{4}/\d{2}/\d{2}/(\d{6})/",
                1
            ),
            1,
            2
        ).cast("int")
    )
)

df_gold_tabela = (
    df_gold_tabela_base
    .groupBy(
        "data",
        "hora"
    )
    .agg(
        F.count("*")
            .alias("qtd_registros_total"),

        F.sum(
            (F.col("registro_com_falha") == 0).cast("int")
        ).alias("qtd_registros_limpos")
    )
    .withColumn(
        "tabela",
        F.lit("ecommerce_pedidos")
    )
    .withColumn(
        "pct_registros_limpos",
        F.round(
            (
                F.col("qtd_registros_limpos")
                / F.col("qtd_registros_total")
            ) * 100,
            2
        )
    )
    .select(
        "data",
        "hora",
        "tabela",
        "qtd_registros_total",
        "qtd_registros_limpos",
        "pct_registros_limpos"
    )
    .orderBy(
        "data",
        "hora"
    )
)

df_gold_tabela.show(50, truncate=False)

In [0]:
# =========================================================
# VALIDAÇÃO — RESUMO POR TABELA
# =========================================================

df_gold_tabela.agg(
    F.sum("qtd_registros_total").alias("total_registros"),
    F.sum("qtd_registros_limpos").alias("total_limpos"),
    F.sum(F.col("pct_registros_limpos").isNull().cast("int"))
        .alias("pct_nulo")
).show()

In [0]:
# =========================================================
# VALIDAÇÃO FINAL — RESUMO POR REGRA
# =========================================================

df_gold_regra.groupBy("regra").agg(
    F.sum("qtd_registros_falhos").alias("total_falhas"),
    F.sum("qtd_registros_total").alias("total_registros")
).orderBy("regra").show()

In [0]:
# =========================================================
# PERSISTÊNCIA IDEMPOTENTE DAS TABELAS GOLD
# =========================================================

(
    df_gold_regra
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "tabela = 'ecommerce_pedidos'")
    .options(**adls_opts)
    .save(gold_regra_path)
)

(
    df_gold_tabela
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "tabela = 'ecommerce_pedidos'")
    .options(**adls_opts)
    .save(gold_tabela_path)
)

print("OK: Gold de ecommerce_pedidos atualizada de forma idempotente.")

In [0]:
# =========================================================
# VALIDAÇÃO FINAL DAS GOLDS DELTA
# =========================================================

df_gold_regra_salva = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(gold_regra_path)
    .filter(F.col("tabela") == "ecommerce_pedidos")
)

df_gold_tabela_salva = (
    spark.read
    .format("delta")
    .options(**adls_opts)
    .load(gold_tabela_path)
    .filter(F.col("tabela") == "ecommerce_pedidos")
)

print("Gold Delta por regra:", df_gold_regra_salva.count())
print("Gold Delta por tabela:", df_gold_tabela_salva.count())

In [0]:
# =========================================================
# PERSISTÊNCIA DAS GOLDS DA DUPLA 4 NO AZURE SQL
# =========================================================

sql_gold_regra = "squad1.dupla4_ecommerce_pedidos_gold_dq_regra"
sql_gold_tabela = "squad1.dupla4_ecommerce_pedidos_gold_dq_tabela"

(
    df_gold_regra
    .write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_hostname)
    .option("database", jdbc_database)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("authentication", "SqlPassword")
    .option("encrypt", "true")
    .option("dbtable", sql_gold_regra)
    .save()
)

(
    df_gold_tabela
    .write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_hostname)
    .option("database", jdbc_database)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("authentication", "SqlPassword")
    .option("encrypt", "true")
    .option("dbtable", sql_gold_tabela)
    .save()
)

print("OK: Golds de ecommerce_pedidos da Dupla 4 gravadas no Azure SQL.")

In [0]:
# =========================================================
# VALIDAÇÃO FINAL DAS GOLDS NO AZURE SQL
# =========================================================

df_sql_gold_regra = spark.read.jdbc(
    url=jdbc_url,
    table=sql_gold_regra,
    properties=connection_properties
)

df_sql_gold_tabela = spark.read.jdbc(
    url=jdbc_url,
    table=sql_gold_tabela,
    properties=connection_properties
)

qtd_regra_esperada = df_gold_regra.count()
qtd_tabela_esperada = df_gold_tabela.count()

qtd_regra_sql = df_sql_gold_regra.count()
qtd_tabela_sql = df_sql_gold_tabela.count()

print(f"Gold por regra  | esperado: {qtd_regra_esperada} | SQL: {qtd_regra_sql}")
print(f"Gold por tabela | esperado: {qtd_tabela_esperada} | SQL: {qtd_tabela_sql}")

assert qtd_regra_sql == qtd_regra_esperada, "Divergência na Gold por regra."
assert qtd_tabela_sql == qtd_tabela_esperada, "Divergência na Gold por tabela."

print("OK: Golds no Azure SQL validadas com sucesso.")

## Observações finais

A camada Gold de `ecommerce_pedidos` consolida métricas de Data Quality para consumo analítico.

- A Gold por regra calcula falhas por regra e por dia do micro-lote.
- A Gold por tabela calcula o percentual de registros limpos por data e hora.
- As tabelas são persistidas em Delta no ADLS.
- As tabelas também são publicadas no Azure SQL para consumo em BI.
- A persistência foi preparada para reexecução sem duplicação dos dados da Dupla 4.